In [2]:
!pip install opencv-python
!pip install opencv-python face-recognition
!pip install numpy==1.26.4

  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.5.4 which is incompatible.


   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.5 MB ? eta -:--:--
    --------------------------------------- 0.3/15.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/15.5 MB 1.1 MB/s eta 0:00:14
   - -------------------------------------- 0.5/15.5 MB 1.1 MB/s eta 0:00:14
   - -------------------------------------- 0.5/15.5 MB 1.1 MB/s eta 0:00:14
   -- ------------------------------------- 0.8/15.5 MB 684.4 kB/s eta 0:00:22
   -- ------------------------------------- 0.8/15.5 MB 684.4 kB/s eta 0:00:22
   -- ------------------------------------- 1.0/15.5 MB 599.0 kB/s eta 0:00:25
   -- ------------------------------------- 1.0/15.5 MB 599.0 kB/s eta 0:00:25
   --- ------------------------------------ 1.3/15.5 MB 583.5 kB/s eta 0:00:25
   --- ------------------------------------ 1.3/15.5 MB 583.5 kB/s eta 0:00:25
   --- --------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
scipy 1.18.0 requires numpy<2.8,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [7]:
import os
import face_recognition

known_encodings = []
known_names = []

for filename in os.listdir("students"):

    if filename.lower().endswith((".jpg", ".jpeg", ".png")):

        path = os.path.join("students", filename)

        image = face_recognition.load_image_file(path)
        encodings = face_recognition.face_encodings(image)

        if len(encodings) > 0:
            known_encodings.append(encodings[0])

            name = os.path.splitext(filename)[0]
            known_names.append(name)

            print(f"Encoded: {name}")

        else:
            print(f"No face found: {filename}")

print("\nTotal students:", len(known_names))

Encoded: Aamir
Encoded: Muhammad Faisal
Encoded: muiz
Encoded: waheed ahmad bhatti
Encoded: Yousaf Safeer

Total students: 5


In [10]:
import cv2
import face_recognition

cap = cv2.VideoCapture(0)

print("Camera started — press Q to quit.")

while True:

    ret, frame = cap.read()

    if not ret:
        print("Camera error.")
        break

    # Resize for faster processing
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)

    # BGR → RGB
    rgb_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    # Detect faces
    locations = face_recognition.face_locations(rgb_frame)

    encodings = face_recognition.face_encodings(
        rgb_frame,
        locations
    )

    for location, encoding in zip(locations, encodings):

        top, right, bottom, left = location

        # Compare with known students
        matches = face_recognition.compare_faces(
            known_encodings,
            encoding,
            tolerance=0.5
        )

        name = "Unknown"

        if True in matches:
            index = matches.index(True)
            name = known_names[index]

        # Scale coordinates back to original frame
        top *= 4
        right *= 4
        bottom *= 4
        left *= 4

        # Green = known, Red = unknown
        color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)

        # Face box
        cv2.rectangle(
            frame,
            (left, top),
            (right, bottom),
            color,
            2
        )

        # Name
        cv2.putText(
            frame,
            name,
            (left, top - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

    cv2.imshow("Student Face Recognition", frame)

    # Q = quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Camera started — press Q to quit.
